<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/DAA_Assign_Sorting_Algorithms_Amazon_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pandas as pd
import numpy as np

In [19]:
# data set ko load krna

df = pd.read_csv("/content/Reviews.csv")
print(f"Original Rows Length : {len(df)}")

Original Rows Length : 568454


In [20]:
# 2. Drop rows with missing values in the important columns
important_cols = ["ProductId", "Score", "Time", "HelpfulnessNumerator", "HelpfulnessDenominator", "Text"]
df = df.dropna(subset=important_cols)
print("After dropping missing values:", len(df))
print("\n")
print(df.head())

# 3. Keep a raw copy of the text
df["RawText"] = df["Text"]

After dropping missing values: 568454


   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought 

In [21]:
df["Text"] = df["Text"].str.replace(r"<[^>]+>", " ", regex=True)
df["Text"] = df["Text"].str.lower()

# 5. Drop rows where the denominator is 0
df = df[df["HelpfulnessDenominator"] != 0]

# 6. Create the ratio column and drop rows where ratio > 1
df["ratio"] = df["HelpfulnessNumerator"] / df["HelpfulnessDenominator"]
df = df[df["ratio"] <= 1]

# 7. Reset the index so rows are numbered 0, 1, 2, ...
df = df.reset_index(drop=True)
print("Final rows:", len(df))

# 8. Make the small sample (first 1000 rows)
S1000 = df.head(1000).copy()
print("S1000 rows:", len(S1000))

# Quick look at the data
print(df[["ProductId", "Score", "Time", "HelpfulnessDenominator", "ratio"]].head())
print(df["Text"][0][:200])

Final rows: 298400
S1000 rows: 1000
    ProductId  Score        Time  HelpfulnessDenominator  ratio
0  B001E4KFG0      5  1303862400                       1    1.0
1  B000LQOCH0      4  1219017600                       1    1.0
2  B000UA0QIQ      2  1307923200                       3    1.0
3  B000E7L2R4      5  1322006400                       1    1.0
4  B0001PB9FE      5  1107820800                       1    1.0
i have bought several of the vitality canned dog food products and have found them all to be of good quality. the product looks more like a stew than a processed meat and it smells better. my labrador


In [22]:
# Switching data from sample to final and vice versa
# data = S1000
data = df

# Part 01 : Sorting Algorithm



# 1.1 Heap Sort and Top-k


In [23]:
import time

# ---------- MAX-HEAP (0-based) ----------
# For index i: left child = 2*i + 1, right child = 2*i + 2
def max_heapify(arr, n, i):
    while True:
        largest = i
        left = 2 * i + 1
        right = 2 * i + 2
        if left < n and arr[left] > arr[largest]:
            largest = left
        if right < n and arr[right] > arr[largest]:
            largest = right
        if largest == i:
            break
        arr[i], arr[largest] = arr[largest], arr[i]
        i = largest

def build_max_heap(arr):
    n = len(arr)
    for i in range(n // 2 - 1, -1, -1):   # start from the last parent
        max_heapify(arr, n, i)

# ---------- MIN-HEAP (used for descending sort) ----------
def min_heapify(arr, n, i):
    while True:
        smallest = i
        left = 2 * i + 1
        right = 2 * i + 2
        if left < n and arr[left] < arr[smallest]:
            smallest = left
        if right < n and arr[right] < arr[smallest]:
            smallest = right
        if smallest == i:
            break
        arr[i], arr[smallest] = arr[smallest], arr[i]
        i = smallest

def build_min_heap(arr):
    n = len(arr)
    for i in range(n // 2 - 1, -1, -1):
        min_heapify(arr, n, i)

# Heap sort with a Min-Heap -> result is DESCENDING
def heap_sort_desc(arr):
    build_min_heap(arr)
    for end in range(len(arr) - 1, 0, -1):
        arr[0], arr[end] = arr[end], arr[0]
        min_heapify(arr, end, 0)


values = data["HelpfulnessDenominator"].astype(int).tolist()

# Quick check that the max-heap works
test = values.copy()
build_max_heap(test)
print("Max-heap root is the biggest:", test[0] == max(values))

# Main task
start = time.time()
heap_sort_desc(values)
end = time.time()

print("Time taken:", round(end - start, 2), "seconds")
print("Top 10 largest HelpfulnessDenominator values:")
print(values[:10])

# Verify with Python's sorted()
check = sorted(data["HelpfulnessDenominator"].astype(int).tolist(), reverse=True)
print("Matches built-in sort:", values == check)

Max-heap root is the biggest: True
Time taken: 1.18 seconds
Top 10 largest HelpfulnessDenominator values:
[923, 878, 815, 593, 569, 562, 562, 562, 562, 562]
Matches built-in sort: True


Why is Heap Sort unstable?
Heap Sort moves values by swapping a parent with a child that may be far away. These long jumps can carry an equal value past its twin. Heap Sort never looks at the original positions, so equal values can change their order.

3-element example: [2a, 2b, 1] (2a and 2b are equal values, with 2a first)

It is already a max-heap.
Swap root with last: [1, 2b, 2a].
Heapify the first 2 items: 1 swaps with 2b, giving [2b, 1, 2a].
Swap root with index 1: [1, 2b, 2a].

# Part 02: Linear Time Sorting

## 2.1 Counting Sort and Product Scores

In [24]:
import time


scores = data["Score"].astype(int).tolist()
n = len(scores)

def counting_sort_by_score(scores):
    n = len(scores)
    k = 5

    # Step 1: count how many times each score appears
    count = [0] * (k + 1)
    for s in scores:
        count[s] += 1

    counts_only = count.copy()

    # Step 2: prefix sums (count[i] = last position + 1 for score i)
    for i in range(1, k + 1):
        count[i] += count[i - 1]

    # Step 3: place items, going BACKWARDS
    output = [0] * n
    for i in range(n - 1, -1, -1):
        s = scores[i]
        count[s] -= 1
        output[count[s]] = i

    return output, counts_only

start = time.time()
sorted_idx, counts = counting_sort_by_score(scores)
end = time.time()

print("Time taken:", round(end - start, 2), "seconds")
print("Counts of each score:")
for s in range(1, 6):
    print("Score", s, ":", counts[s])


sorted_data = data.iloc[sorted_idx]
print("First 10 scores after sorting:", sorted_data["Score"].head(10).tolist())
print("Last 10 scores after sorting:", sorted_data["Score"].tail(10).tolist())


check_idx = sorted(range(n), key=lambda i: scores[i])
print("Matches built-in stable sort:", sorted_idx == check_idx)

Time taken: 0.05 seconds
Counts of each score:
Score 1 : 40002
Score 2 : 19165
Score 3 : 24217
Score 4 : 38638
Score 5 : 176378
First 10 scores after sorting: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Last 10 scores after sorting: [5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
Matches built-in stable sort: True


**Why must the placement loop run backwards?**

The prefix sum gives the last free spot of each score. When we go backwards, the last review with that score takes the last spot, and the earlier ones fill spots before it. So equal scores keep their original order, which is stability.

**What if we run it forwards?**

The first review of a score takes the last spot, so equal reviews end up in reverse order. The sort still sorts correctly, but it is no longer stable.

Example: two reviews with the same score 3: [3a, 3b].

**Backwards:** 3b goes to the last spot, 3a goes before it.
**Result:** 3a, 3b (stable).

**Forwards:** 3a goes to the last spot, 3b goes before it. **Result**: 3b, 3a (unstable).

**Why is Counting Sort a terrible idea for Unix timestamps?**

Counting Sort needs a count array as big as the range of values. Timestamps in this data run from about 940 million to 1.35 billion, so even after subtracting the minimum, the range is hundreds of millions. We would need a huge array (many GB of memory) just to sort about 500,000 reviews. The time is O(n + k), and here k is far bigger than n, so it is slow and wastes memory.

## Radix Sort on Timestamps

In [25]:
import time

# Get the Time column as a simple list
times = data["Time"].astype(int).tolist()
n = len(times)

def radix_sort(keys, base):
    n = len(keys)
    order = list(range(n))
    biggest = max(keys)
    exp = 1
    passes = 0

    while biggest // exp > 0:

        count = [0] * base
        for i in order:
            digit = (keys[i] // exp) % base
            count[digit] += 1

        for d in range(1, base):
            count[d] += count[d - 1]

        output = [0] * n
        for j in range(n - 1, -1, -1):    # BACKWARDS keeps it stable
            i = order[j]
            digit = (keys[i] // exp) % base
            count[digit] -= 1
            output[count[digit]] = i

        order = output
        exp *= base
        passes += 1

    return order, passes

# ---- Base 10 ----
start = time.time()
idx10, passes10 = radix_sort(times, 10)
t10 = time.time() - start

# ---- Base 256 ----
start = time.time()
idx256, passes256 = radix_sort(times, 256)
t256 = time.time() - start

print("Base-10  : passes =", passes10, ", time =", round(t10, 2), "seconds")
print("Base-256 : passes =", passes256, ", time =", round(t256, 2), "seconds")

# ---- Verify with Python's stable sort ----
check = sorted(range(n), key=lambda i: times[i])
print("Base-10 matches built-in sort :", idx10 == check)
print("Base-256 matches built-in sort:", idx256 == check)

# Show the first 5 sorted timestamps
sorted_data = data.iloc[idx256]
print("First 5 times:", sorted_data["Time"].head(5).tolist())
print("Last 5 times :", sorted_data["Time"].tail(5).tolist())

Base-10  : passes = 10 , time = 1.54 seconds
Base-256 : passes = 4 , time = 0.66 seconds
Base-10 matches built-in sort : True
Base-256 matches built-in sort: True
First 5 times: [940809600, 944438400, 944438400, 944438400, 947376000]
Last 5 times : [1351123200, 1351123200, 1351123200, 1351209600, 1351209600]


**Does fewer passes always mean faster?**

No. Fewer passes means fewer trips over the data, but each pass scatters items into base different places. Base-256 needs a count array of only 256 numbers, which easily fits in the CPU cache. If we went bigger (for example Base-65536), the count array and the 65,536 write positions would no longer fit in the cache. Then almost every write is a cache miss, and the pass becomes slow. So the best base depends on the machine’s cache size. On a machine with a small cache, a smaller base can win.

## 2.3: Bucket Sort on Helpfulness Ratio

In [26]:
import time

# Get the ratio column as a simple list
ratios = data["ratio"].tolist()
n = len(ratios)

# Insertion sort on a list of row numbers (sorted by their ratio)
def insertion_sort(bucket):
    for j in range(1, len(bucket)):
        key_row = bucket[j]
        key_val = ratios[key_row]
        k = j - 1
        while k >= 0 and ratios[bucket[k]] > key_val:
            bucket[k + 1] = bucket[k]
            k -= 1
        bucket[k + 1] = key_row

def bucket_sort(ratios):
    n = len(ratios)
    num_buckets = 10

    # Step 1: make 10 empty buckets
    buckets = [[] for _ in range(num_buckets)]

    # Step 2: put each row in its bucket (0.0-0.1 -> bucket 0, ... 0.9-1.0 -> bucket 9)
    for i in range(n):
        b = int(ratios[i] * num_buckets)
        if b == num_buckets:
            b = num_buckets - 1
        buckets[b].append(i)

    sizes = [len(b) for b in buckets]

    # Step 3: insertion sort inside each bucket
    for b in buckets:
        insertion_sort(b)

    # Step 4: join the buckets in order
    result = []
    for b in buckets:
        result.extend(b)

    return result, sizes

start = time.time()
sorted_idx, sizes = bucket_sort(ratios)
end = time.time()

print("Time taken:", round(end - start, 2), "seconds")
print("Bucket sizes:")
for b in range(10):
    print("Bucket", b, "(", b / 10, "to", (b + 1) / 10, "):", sizes[b])

# Check that the ratios are in sorted order
sorted_ratios = [ratios[i] for i in sorted_idx]
print("Is sorted:", sorted_ratios == sorted(ratios))
print("First 5 ratios:", sorted_ratios[:5])
print("Last 5 ratios :", sorted_ratios[-5:])

Time taken: 85.59 seconds
Bucket sizes:
Bucket 0 ( 0.0 to 0.1 ): 34518
Bucket 1 ( 0.1 to 0.2 ): 2248
Bucket 2 ( 0.2 to 0.3 ): 4353
Bucket 3 ( 0.3 to 0.4 ): 6246
Bucket 4 ( 0.4 to 0.5 ): 2748
Bucket 5 ( 0.5 to 0.6 ): 23166
Bucket 6 ( 0.6 to 0.7 ): 13916
Bucket 7 ( 0.7 to 0.8 ): 9548
Bucket 8 ( 0.8 to 0.9 ): 12112
Bucket 9 ( 0.9 to 1.0 ): 189545
Is sorted: True
First 5 ratios: [0.0, 0.0, 0.0, 0.0, 0.0]
Last 5 ratios : [1.0, 1.0, 1.0, 1.0, 1.0]


**Why is one bucket so big?**

Most people who vote on helpfulness vote the same way. Reviews with 1 out of 1 or 2 out of 2 votes have ratio exactly 1.0. Many others have exactly 0.0 or 0.5. Human votes are not spread evenly from 0 to 1. They pile up on a few values.

**How does this affect O(n)?**

Bucket Sort is O(n) only when the data is uniform, so every bucket gets about n/10 items and the buckets are small. With a spike, one bucket holds a large part of the data. Insertion Sort on a bucket of size m costs O(m²). So the total time moves toward O(n²), and the “linear time” benefit is lost. A fix is to use Merge Sort inside big buckets, which gives O(m log m).

# Part 3: String Searching

## 3.1: Naive Search vs. KMP

In [27]:

texts = data["Text"].head(10000).tolist()
pattern = "highly recommend"

# ---------- Naive search ----------
def naive_search(text, pattern):
    n, m = len(text), len(pattern)
    comps = 0
    matches = 0
    for i in range(n - m + 1):
        j = 0
        while j < m:
            comps += 1
            if text[i + j] != pattern[j]:
                break
            j += 1
        if j == m:
            matches += 1
    return matches, comps

# ---------- LPS array ----------
def build_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length = 0
    i = 1
    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]
        else:
            lps[i] = 0
            i += 1
    return lps

# ---------- KMP search ----------
def kmp_search(text, pattern, lps):
    n, m = len(text), len(pattern)
    comps = 0
    matches = 0
    i = 0
    j = 0
    while i < n:
        comps += 1                           # one character comparison
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == m:
                matches += 1
                j = lps[j - 1]
        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return matches, comps

# ---------- Run on all reviews ----------
lps = build_lps(pattern)

naive_matches, naive_comps = 0, 0
kmp_matches, kmp_comps = 0, 0

for t in texts:
    m1, c1 = naive_search(t, pattern)
    naive_matches += m1
    naive_comps += c1
    m2, c2 = kmp_search(t, pattern, lps)
    kmp_matches += m2
    kmp_comps += c2

print("Reviews searched:", len(texts))
print("Naive -> matches:", naive_matches, ", comparisons:", naive_comps)
print("KMP   -> matches:", kmp_matches, ", comparisons:", kmp_comps)
print("Same matches:", naive_matches == kmp_matches)

# Verify with Python's count (allowed for checking)
check = sum(t.count(pattern) for t in texts)
print("Matches built-in count:", check == naive_matches)

# ---------- LPS check for the concept question ----------
print("LPS of 'really really good':", build_lps("really really good"))

Reviews searched: 10000
Naive -> matches: 373 , comparisons: 4807721
KMP   -> matches: 373 , comparisons: 4927430
Same matches: True
Matches built-in count: True
LPS of 'really really good': [0, 0, 0, 0, 0, 0, 0, 1, 2, 3, 4, 5, 6, 7, 0, 0, 0, 0]


**Why is natural language forgiving to Naive search?**

Naive search is slow only when it matches a long part of the pattern and then fails, again and again. This needs very repetitive text, like aaaaaaab. In English, most start positions fail on the first or second character. For example, text that starts with t or s does not match h in “highly” immediately. So each position costs about 1 comparison, and the total is close to n, the same as KMP. The O(n × m) worst case almost never happens in real reviews.

## 3.2: Rabin-Karp and Spurious Hits

In [28]:
import time

# Join reviews into one long text and keep the first 100,000 characters
big_text = " ".join(data["Text"].tolist())[:100000]
pattern = "gluten free"
print("Text length:", len(big_text))

def rabin_karp(text, pattern, q):
    n, m = len(text), len(pattern)
    d = 256
    h = pow(d, m - 1, q)

    # Step 1: hash of the pattern and of the first window
    p_hash = 0
    t_hash = 0
    for i in range(m):
        p_hash = (d * p_hash + ord(pattern[i])) % q
        t_hash = (d * t_hash + ord(text[i])) % q

    hash_hits = 0
    verified = 0
    spurious = 0

    # Step 2: slide the window over the text
    for i in range(n - m + 1):
        if p_hash == t_hash:
            hash_hits += 1
            if text[i:i + m] == pattern:
                verified += 1
            else:
                spurious += 1

        # Step 3: rolling hash -> remove the first char, add the next char
        if i < n - m:
            t_hash = (d * (t_hash - ord(text[i]) * h) + ord(text[i + m])) % q

    return hash_hits, verified, spurious

for q in [101, 1000000007]:
    start = time.time()
    hits, ver, spur = rabin_karp(big_text, pattern, q)
    end = time.time()
    print("\nq =", q)
    print("Hash hits       :", hits)
    print("Verified matches:", ver)
    print("Spurious hits   :", spur)
    print("Time            :", round(end - start, 3), "seconds")

# Verify with Python's count (allowed for checking)
print("\nBuilt-in count:", big_text.count(pattern))

Text length: 100000

q = 101
Hash hits       : 929
Verified matches: 0
Spurious hits   : 929
Time            : 0.027 seconds

q = 1000000007
Hash hits       : 0
Verified matches: 0
Spurious hits   : 0
Time            : 0.036 seconds

Built-in count: 0


**Why does a small q give more spurious hits?**

With q = 101, there are only 101 possible hash values. A random window matches the pattern hash by chance about 1 time in 101. A huge q makes that chance about 1 in a billion.

**Why is summing ASCII values a bad hash?**

Order is ignored. “listen” and “silent” have the same letters, so they give the same sum and always collide. Any anagram does the same.

**Different strings can also have the same sum:**

“ad” (97+100) and “bc” (98+99) both give 197.
The sums are small and bunched together.
For an 11-letter phrase the sum is only about 1,000, so many windows share it. This gives huge numbers of spurious hits, and Rabin-Karp slows toward O(n × m).

Our polynomial hash multiplies each character by a power of the base, so position matters and “listen” and “silent” get different hashes.

# Part 04 : Technique Integration

## 4.1: The "Gluten Free" Mini-Pipeline


In [29]:
import time

# ---------- Settings ----------
MIN_REVIEWS = 50
pattern = "gluten free"

texts = data["Text"].tolist()
product_ids = data["ProductId"].tolist()
n = len(texts)

# ---------- KMP (from Task 3.1) ----------
def build_lps(pattern):
    m = len(pattern)
    lps = [0] * m
    length = 0
    i = 1
    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]
        else:
            lps[i] = 0
            i += 1
    return lps

def kmp_contains(text, pattern, lps):
    n, m = len(text), len(pattern)
    i = 0
    j = 0
    while i < n:
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == m:
                return True
        else:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1
    return False

# ---------- Step 1: flag reviews with "gluten free" ----------
start = time.time()
lps = build_lps(pattern)
flags = [kmp_contains(t, pattern, lps) for t in texts]
print("Step 1 done. Reviews with 'gluten free':", sum(flags),
      "| time:", round(time.time() - start, 2), "s")

# ---------- Step 2: Radix Sort on ProductId  ----------
def radix_sort_strings(keys):
    n = len(keys)
    order = list(range(n))
    max_len = max(len(k) for k in keys)


    for pos in range(max_len - 1, -1, -1):
        count = [0] * 257
        for i in order:
            k = keys[i]
            c = ord(k[pos]) + 1 if pos < len(k) else 0
            count[c] += 1

        for d in range(1, 257):
            count[d] += count[d - 1]

        output = [0] * n
        for j in range(n - 1, -1, -1):
            i = order[j]
            k = keys[i]
            c = ord(k[pos]) + 1 if pos < len(k) else 0
            count[c] -= 1
            output[count[c]] = i
        order = output
    return order

start = time.time()
sorted_idx = radix_sort_strings(product_ids)
print("Step 2 done. Radix sort time:", round(time.time() - start, 2), "s")


sorted_ids = [product_ids[i] for i in sorted_idx]
print("ProductIds sorted correctly:", sorted_ids == sorted(product_ids))

# ---------- Step 3: group by product and compute percentages ----------
results = []
i = 0
while i < n:
    current = sorted_ids[i]
    total = 0
    gluten = 0

    while i < n and sorted_ids[i] == current:
        total += 1
        if flags[sorted_idx[i]]:
            gluten += 1
        i += 1
    if total >= MIN_REVIEWS:
        results.append((gluten / total * 100, current, gluten, total))

print("Products with at least", MIN_REVIEWS, "reviews:", len(results))

# ---------- Final: top 10 (built-in sort is allowed here) ----------
results.sort(reverse=True)
print("\nTop 10 products by % of reviews containing 'gluten free':")
print("Rank | ProductId  | Percentage | Gluten reviews / Total")
for rank in range(min(10, len(results))):
    pct, pid, g, t = results[rank]
    print(rank + 1, "   |", pid, "|", round(pct, 2), "% |", g, "/", t)

# ---------- Verify with pandas (allowed for checking) ----------
check = data.assign(flag=flags).groupby("ProductId")["flag"].agg(["sum", "count"])
check = check[check["count"] >= MIN_REVIEWS]
check["pct"] = check["sum"] / check["count"] * 100
print("\nPandas check (top 10 percentages):")
print(check["pct"].sort_values(ascending=False).head(10).round(2).tolist())

Step 1 done. Reviews with 'gluten free': 4083 | time: 12.43 s
Step 2 done. Radix sort time: 1.1 s
ProductIds sorted correctly: True
Products with at least 50 reviews: 945

Top 10 products by % of reviews containing 'gluten free':
Rank | ProductId  | Percentage | Gluten reviews / Total
1    | B002AQP5FW | 46.67 % | 28 / 60
2    | B001D0676C | 38.67 % | 29 / 75
3    | B000EVG8H4 | 36.7 % | 40 / 109
4    | B004391DK0 | 35.45 % | 39 / 110
5    | B000FK63IS | 33.96 % | 18 / 53
6    | B001L4JH5I | 32.26 % | 30 / 93
7    | B000NMJWZO | 31.79 % | 62 / 195
8    | B0040K41MY | 29.31 % | 17 / 58
9    | B000K8UL4C | 28.57 % | 20 / 70
10    | B000DZFMEQ | 28.33 % | 34 / 120

Pandas check (top 10 percentages):
[46.67, 38.67, 36.7, 35.45, 33.96, 32.26, 31.79, 29.31, 28.57, 28.33]


Sorting by ProductId puts all reviews of one product next to each other, so one pass counts them. The percentage is gluten reviews divided by total reviews. The 50-review rule stops a product with 1 of 1 reviews from showing 100%.